DATA EXPLORATION

In [1]:
import pandas as pd

EXPLORING SMSSPAMCOLLECTION

In [26]:
rows = []
with open("../data/raw/SMSSpamCollection.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        label, text = line.split(maxsplit=1)
        rows.append((label, text))

df = pd.DataFrame(rows, columns=["label", "text"])

In [31]:
df.head(10)

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [28]:
df.shape

(5574, 2)

In [29]:
df["label"].value_counts()

label
ham     4827
spam     747
Name: count, dtype: int64

In [30]:
df["text"].isna().sum() 

np.int64(0)

The dataset contains 5,574 ENGLISH messages. It is moderately imbalanced since there is a noticeable difference between the amount 'spam' and 'ham' messages. There are no empty texts.

CLEANING AND SPLITTING

In [32]:
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Remove exact duplicate rows before splitting to prevent data leakage.
clean_df = df.drop_duplicates(subset=["label", "text"]).reset_index(drop=True)

train_df, remaining_df = train_test_split(
    clean_df,
    test_size=0.30,
    stratify=clean_df["label"],
    random_state=RANDOM_STATE,
)
validation_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    stratify=remaining_df["label"],
    random_state=RANDOM_STATE,
)

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
train_df.to_csv(processed_dir / "train.csv", index=False)
validation_df.to_csv(processed_dir / "validation.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

assert len(train_df) + len(validation_df) + len(test_df) == len(clean_df)
assert not set(train_df.index) & set(validation_df.index)
assert not set(train_df.index) & set(test_df.index)
assert not set(validation_df.index) & set(test_df.index)

print(f"Removed {len(df) - len(clean_df)} exact duplicate rows.")
print(f"Saved {len(train_df)} train, {len(validation_df)} validation, and {len(test_df)} test rows.")
print("Label proportions:")
print(pd.DataFrame({
    "train": train_df["label"].value_counts(normalize=True),
    "validation": validation_df["label"].value_counts(normalize=True),
    "test": test_df["label"].value_counts(normalize=True),
}).round(3))

Removed 406 exact duplicate rows.
Saved 3617 train, 775 validation, and 776 test rows.
Label proportions:
       train  validation   test
label                          
ham    0.876       0.876  0.875
spam   0.124       0.124  0.125
